# 🎬 Wan2.1 Text-to-Video Studio (Colab T4 Free)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/colab/04_wan_ltx_video_studio.ipynb)
[![Notebook Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **Real weights, real inference:** [`Wan-AI/Wan2.1-T2V-1.3B-Diffusers`](https://huggingface.co/Wan-AI/Wan2.1-T2V-1.3B-Diffusers)
> (Apache-2.0) through the official diffusers `WanPipeline`. The earlier version of this notebook only
> rendered procedural colour frames — that is fixed now; if the model cannot load you get an explicit
> error instead of a fake video.

**What to expect on a free Tesla T4 (16 GB):**

| Setting | Value |
| :--- | :--- |
| Resolution | 480 × 832 |
| Frames | 33 (≈ 2 s @ 16 fps) — must be `4k + 1` |
| Precision | FP16 transformer + FP32 VAE |
| Offload | `enable_model_cpu_offload()` keeps the T5 encoder on CPU so the run fits in 15 GB |
| Time per clip | **roughly 10–25 minutes** on a free T4 (first run also downloads ~13 GB of weights) |

The Gradio link is password protected (Basic auth). `LTX-Video` is **not** used by this notebook.


In [ ]:
#@title 🚀 1. Hardware Check & Free VRAM
import hashlib, os, socket, time, uuid

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

raw_fp = f"colab-video-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "colab-video-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]
print(f"🆔 Unique Session Fingerprint: {INSTANCE_ID}")

import torch
if not torch.cuda.is_available():
    raise SystemExit("❌ No GPU. In Colab: Runtime -> Change runtime type -> T4 GPU.")
free, total = torch.cuda.mem_get_info()
print(f"🔥 {torch.cuda.get_device_name(0)} — {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")


In [ ]:
#@title 📦 2. Install Wan2.1 Video Stack
# WanPipeline needs diffusers >= 0.33.0.
!pip install -q --upgrade "diffusers>=0.33.0" transformers accelerate ftfy "imageio[ffmpeg]" gradio
print("✅ Video dependencies installed.")


In [ ]:
#@title 🧠 3. Load Wan2.1 1.3B Pipeline
import os
import torch
from diffusers import AutoencoderKLWan, WanPipeline

MODEL_ID = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
OUTPUT_DIR = "/content" if os.path.isdir("/content") else "/tmp"

print("📥 Loading VAE (FP32) + transformer (FP16) + T5 encoder ...")
vae = AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.float16)

# Keeps the large T5 text encoder on CPU between steps: this is what makes 15 GB fit.
pipe.enable_model_cpu_offload()

if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"✅ Pipeline ready — {free / 1024**3:.1f} GB VRAM still free")


In [ ]:
#@title 🎬 4. Generate a Video (password protected)
import os
import time

import gradio as gr
import torch
from diffusers.utils import export_to_video

NEGATIVE_PROMPT = (
    "Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, "
    "static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, "
    "extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, "
    "fused fingers, still picture, messy background, three legs, many people in the background, walking backwards"
)


def generate_video(prompt: str, negative_prompt: str, steps: int, frames: int, seed: int):
    if not prompt.strip():
        raise gr.Error("Please enter a video prompt.")
    frames = int(frames)
    if (frames - 1) % 4 != 0:
        raise gr.Error("Frame count must follow the 4k+1 rule (17, 33, 49 ...).")

    generator = torch.Generator(device="cpu").manual_seed(int(seed))
    started = time.time()
    try:
        result = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt or None,
            height=480,
            width=832,
            num_frames=frames,
            num_inference_steps=int(steps),
            guidance_scale=5.0,
            generator=generator,
        )
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise gr.Error(f"CUDA OOM — reduce frames/steps or restart the runtime. ({exc})")

    out_path = os.path.join(OUTPUT_DIR, f"wan_{int(time.time())}.mp4")
    export_to_video(result.frames[0], out_path, fps=16)
    print(f"🎬 {frames} frames in {time.time() - started:.0f}s -> {out_path}")
    return out_path


def studio_auth():
    """Password for the public Gradio link (Basic auth).

    Priority: STUDIO_PASSWORD -> Colab secret -> Kaggle secret -> random one-time password.
    """
    import os
    import secrets

    password = os.getenv("STUDIO_PASSWORD", "")
    if not password:
        try:
            from google.colab import userdata
            password = userdata.get("STUDIO_PASSWORD") or ""
        except Exception:
            password = ""
    if not password:
        try:
            from kaggle_secrets import UserSecretsClient
            password = UserSecretsClient().get_secret("STUDIO_PASSWORD") or ""
        except Exception:
            password = ""
    if not password:
        password = secrets.token_urlsafe(9)
        print(f"\n🔑 One-time password for this session: {password}\n")
    return ("admin", password)

demo = gr.Interface(
    fn=generate_video,
    inputs=[
        gr.Textbox(label="Prompt", lines=3,
                   value="A sleek sports car cruising down a neon-lit Tokyo highway at night, rain reflections, cinematic"),
        gr.Textbox(label="Negative prompt", lines=2, value=NEGATIVE_PROMPT),
        gr.Slider(minimum=10, maximum=40, value=20, step=1, label="Inference steps"),
        gr.Slider(minimum=17, maximum=49, value=33, step=16, label="Frames (4k+1)"),
        gr.Number(value=42, label="Seed"),
    ],
    outputs=gr.Video(label="Generated video (mp4, 16 fps)"),
    title="🎬 Wan2.1 T2V 1.3B — Colab Free T4",
    description="Real Wan2.1 inference. A 33-frame clip takes roughly 10–25 min on a free T4.",
)

demo.queue().launch(share=True, auth=studio_auth(), debug=False)
